In [ ]:
import pandas as pd
import math
import os

from scipy import stats
from numpy import random
import numpy as np

from hag.models import tanh
activation_function = lambda x : tanh(x)

SEED = 923984

# Loading and Preprocessing

Lots of different on availabale : https://towardsdatascience.com/a-data-lakes-worth-of-audio-datasets-b45b88cd4ad

Regression : http://tseregression.org/ + https://arxiv.org/pdf/2012.02974

Prediction Datasets available :

* MackeyGlass
* Lorenz
* Sunspot

Classification Datasets available :

* Custom :  FSDD, JapaneseVowels
* Aeon : SpokenArabicDigits, CatsDogs, LSST
* Torchaudio: SPEECHCOMMANDS

More on https://www.timeseriesclassification.com/dataset.php or https://pytorch.org/audio/stable/datasets.html

# Evaluation functions

In [ ]:
from hag.analysis.commons import load_data, evaluate_dataset_on_test_rnn, evaluate_dataset_on_test

# Visualization

In [ ]:
from hag.analysis.commons import function_colors, functions_order, function_mapping, dataset_label_map

# Test scores

In [ ]:
import numpy as np

from datetime import datetime

from hag.performances import retrieve_best_model

# Create an empty DataFrame to store the results
columns = ['Dataset', 'Function', 'Average Score', 'Standard Deviation', 'Date']
variate_type = "multi"  # "multi" or "uni"


for dataset_name in ["CatsDogs", "FSDD", "JapaneseVowels"]:
    new_results = pd.DataFrame(columns=columns)
    # Can be "MackeyGlass", "Lorenz", "Sunspot_daily", "CatsDogs", "JapaneseVowels", "FSDD", "SpokenArabicDigits", "SPEECHCOMMANDS"
    spectral_representation = "mfcc" if dataset_name in ["CatsDogs", "JapaneseVowels", "FSDD", "SpokenArabicDigits", "SPEECHCOMMANDS"] else "stft"
    pretrain_data, train_data, test_data, Y_train, Y_test, is_multivariate, is_instances_classification = load_data(dataset_name, spectral_representation, visualize=True)
    if is_instances_classification:
        file_name = "outputs/test_results/test_results_classification.csv"
    else: 
        file_name = "outputs/test_results/test_results_prediction.csv"
    print(dataset_name)
    # Simulate your data and loop for evaluation
    
    # "random_ee", "random_ei", "diag_ee", "diag_ei", "ip_correct", "anti-oja_fast",  "ip-anti-oja_fast", "mean_hag", "var_hag", "lstm_last", "rnn", "gru", "short-hag", "hsp"
    for function_name in ["short-hag", "hsp"]:
        print(function_name)

        # dispatch to the correct evaluator
        if function_name in ["lstm_last", "rnn", "rnn-mean_hag", "gru"]:
            prefix = "lstm_tpe" if dataset_name == "SPEECHCOMMANDS" else "tpe"
            study = retrieve_best_model(function_name, dataset_name, is_multivariate, variate_type = "multi", data_type = "normal", prefix="tpe", db_dir="hag/hpo/legacy_studies")
            scores = evaluate_dataset_on_test_rnn(
                study,
                dataset_name,
                function_name,
                pretrain_data,
                train_data,
                test_data,
                Y_train,
                Y_test,
                is_instances_classification,
                nb_trials=8,
                record_metrics=False
            )
        else:
            study = retrieve_best_model(function_name, dataset_name, is_multivariate, variate_type = "multi", data_type = "normal", prefix="tpe", db_dir="hag/hpo/legacy_studies")

            scores = evaluate_dataset_on_test(
                study,
                dataset_name,
                function_name, 
                pretrain_data, 
                train_data, 
                test_data,
                Y_train,
                Y_test,
                is_instances_classification,
                nb_trials = 8,
                record_metrics=False
            )
            
        # Compute the average and standard deviation of the scores
        average_score = np.mean(scores)
        std_deviation = np.std(scores)
    
        if is_instances_classification:
            formatted_average = f"{round(average_score * 100, 5)} %"
            formatted_std = f"± {round(std_deviation * 100, 5)} %"
        else:
            formatted_average = f"{round(average_score, 5)}"
            formatted_std = f"± {round(std_deviation, 5)}"
        
        # Capture the current date
        current_date = datetime.now().strftime('%Y-%m-%d')
        
        # Create a new DataFrame row with the Date column
        new_row = pd.DataFrame({
            'Dataset': [dataset_name],
            'Function': [function_name],
            'Average Score': [formatted_average],
            'Standard Deviation': [formatted_std],
            'Date': [current_date]
        })
        
        # Concatenate the new row to the results DataFrame
        new_results = pd.concat([new_results, new_row], ignore_index=True)
    
    
    # Display the DataFrame
    print(new_results)
    
    # Load the existing CSV
    if os.path.exists(file_name):
        previous_results = pd.read_csv(file_name)
    else:
        columns = ['Dataset', 'Function', 'Average Score', 'Standard Deviation', 'Date']
        previous_results = pd.DataFrame(columns=columns)
        previous_results.to_csv(file_name, index=False)
        print(f"{file_name} created successfully.")
        
    tots_results = pd.concat([new_results, previous_results], axis=0)
    
    tots_results.to_csv(file_name, index=False)
    print(f"Results saved to {file_name}.")

## Visualisation

In [ ]:
import os
import pandas as pd 

file_name = "outputs/test_results/test_results_classification.csv"

if 'file_name' not in locals() and 'file_name' not in globals():
    file_name = "outputs/test_results/test_results_prediction.csv"  #  test_results_classification.csv or test_results_prediction.csv

if os.path.exists(file_name):
    previous_results = pd.read_csv(file_name)
else:
    # File does not exist, create it with the necessary columns
    columns = ['Dataset', 'Function', 'Average Score', 'Standard Deviation', 'Date']
    previous_results = pd.DataFrame(columns=columns)
    # Save the empty DataFrame as a CSV
    previous_results.to_csv(file_name, index=False)
    print(f"{file_name} created successfully.")

print(f"Results saved to {file_name}.")
previous_results

In [ ]:
import pandas as pd

from matplotlib import pyplot as plt
import numpy as np

all_results = pd.read_csv(file_name)
df = pd.DataFrame(all_results)

# Clean data as before
df['Average Score'] = df['Average Score'].astype(str).str.replace('%', '').astype(float)
df['Standard Deviation'] = df['Standard Deviation'].str.replace('±', '').str.replace('%', '').astype(float)

df = df[df['Function'] != 'ip']

df['Function'] = df['Function'].map(function_mapping)

# Optional replacements for dataset names
#df = df[df['Dataset'].isin(["Lorenz", "MackeyGlass", "Sunspot"])]

if file_name == "test_results/test_results_classification.csv":
    df['Dataset'] = df['Dataset'].str.replace('SpokenArabicDigits', 'Spoken\nArabic\nDigits')
    df['Dataset'] = df['Dataset'].str.replace('SPEECHCOMMANDS', 'SPEECH\nCOMMANDS')
    df['Dataset'] = df['Dataset'].str.replace('JapaneseVowels', 'Japanese\nVowels')


fig, ax = plt.subplots(figsize=(16, 6))

datasets = df['Dataset'].unique()
x = np.arange(len(datasets))  # The label locations
width = 0.1                   # Width of each bar

for i, func in enumerate(functions_order):
    # Grab only rows for this function
    values = df[df['Function'] == func]
    
    # We create a Series in the same order as 'datasets'
    merged = pd.DataFrame({'Dataset': datasets}).merge(values, on='Dataset', how='left')
    
    ax.bar(
        x + i * width,
        merged['Average Score'],
        width,
        label=func,
        yerr=merged['Standard Deviation'],
        capsize=5,
        color=function_colors[func],
#        log=True
    )

fontsize = 14

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', labelsize=fontsize)

if file_name == "test_results/test_results_prediction.csv":
    plt.ylabel('NRMSE', size=fontsize)
else:
    plt.ylabel('Classification Rate', size=fontsize)
plt.legend(title='Algorithm', fontsize=10, title_fontsize=fontsize, bbox_to_anchor=(0.5, 1.15), loc='upper center', ncol=len(functions_order))

# Position x-ticks in the center of all the bars for each dataset
ax.set_xticks(x + width * (len(functions_order)-1)/2)
ax.set_xticklabels(datasets)

plt.tight_layout()
plt.show()

# Export best parameters

In [ ]:

import pandas as pd
from hag.performances import retrieve_best_model
# Can be "MackeyGlass", "Lorenz", "Sunspot_daily", "CatsDogs", "JapaneseVowels", "FSDD", "SpokenArabicDigits", "SPEECHCOMMANDS"
datasets = [
   "CatsDogs", "JapaneseVowels", "FSDD"
]

# helper ────────────────────────────────────────────────────────────
def smart_format(x, ndigits=5):
    """
    • If |x| is smaller than 1e-3  → scientific notation with `ndigits` decimals.
    • Otherwise                  → round to `ndigits` decimals (keeps 12345.6, 0.012345 …).
    """
    if not isinstance(x, float):
        return x                      # leave non-floats untouched
    if x == 0.0:
        return 0.0                    # keep plain zero
    if abs(x) < 1e-3:                 # 0.000 … region
        return f"{x:.{ndigits}e}"     # e.g. 1.23456e-04
    return round(x, ndigits)          # ordinary decimal

# main loop ─────────────────────────────────────────────────────────
 # "random_ee", "random_ei", "ip_correct", "anti-oja_fast",  "ip-anti-oja_fast", "hadsp", "desp", "lstm_last", "gru", "short-hag", "hsp"
for fn_name in ["short-hag", "hsp"]:
    rows = []
    for ds in datasets:
        prefix = "lstm_tpe" if (ds == "SPEECHCOMMANDS" and fn_name in ["lstm_last", "gru"]) else "tpe"
        study = retrieve_best_model(function_name, dataset_name, is_multivariate, variate_type = "multi", data_type = "normal", prefix=prefix, db_dir="hag/hpo/legacy_studies")

        params = {k: smart_format(v) for k, v in study.best_trial.params.items()}

        rows.append({
            "dataset": ds,
            "function_name": fn_name,
            **params,
        })

    pd.DataFrame(rows).to_csv(
        f"outputs/best_hyperparameters_{fn_name}.csv",
        index=False
    )
    print(f"Results saved to best_hyperparameters_{fn_name}.csv")

In [ ]:
import os
import pandas as pd
from hag.performances import retrieve_best_model

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
DATASETS = [
    "JapaneseVowels", "CatsDogs", "FSDD",
]

#     "random_ee", "random_ei", "ip_correct", "anti-oja_fast", "ip-anti-oja_fast", "hadsp", "desp", "lstm_last", "gru", "short-hag", "hsp"
FUNCTIONS = [
    "short-hag", "hsp",
]

OUTPUT_DIR = "outputs/best_hyperparameters"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "best_cv_scores.csv")

# ------------------------------------------------------------------
# Core collection loop
# ------------------------------------------------------------------
rows = []

for fn_name in FUNCTIONS:
    for ds_name in DATASETS:
        # Grab the Optuna study that already contains the finished HPO
        prefix = "lstm_tpe" if (ds_name == "SPEECHCOMMANDS" and fn_name in ["lstm_last", "gru"]) else "new_tpe"
        study = retrieve_best_model(
            fn_name, ds_name,
            is_multivariate=True, variate_type="multi", data_type="normal", prefix=prefix
        )

        # Optuna exposes the optimum either as study.best_value or study.best_trial.value
        best_score = getattr(study, "best_value", study.best_trial.value)

        rows.append({
                "dataset": ds_name,
                "function": fn_name,
                "best_score": best_score,
        })

# ------------------------------------------------------------------
# Save to disk
# ------------------------------------------------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.DataFrame(rows).to_csv(OUTPUT_FILE, index=False)
print(f"Results saved to {OUTPUT_FILE}")

# Richness

In [ ]:
import pandas as pd
from hag.performances import retrieve_best_model

# Create an empty DataFrame to store the results
columns = [
    "dataset", 
    "function_name", 
    "spectral_radius_mean", 
    "spectral_radius_std", 
    "pearson_mean", 
    "pearson_std",
    "CEVD_mean",
    "CEVD_std",
    "dcor_mean",
    "dcor_std",
    "final_correlations_mean",
    "final_correlations_std",
]


# List of datasets (extract from filenames)
datasets = [
    "JapaneseVowels",
    "CatsDogs",
    "FSDD",
 #   "SpokenArabicDigits",
 #   "SPEECHCOMMANDS",
#    "MackeyGlass",
#    "Lorenz",
#    "Sunspot_daily",
]


new_results = []
for dataset in datasets:
    print(dataset)
    spectral_representation = "mfcc" if dataset in ["CatsDogs", "FSDD", "JapaneseVowels", "SPEECHCOMMANDS", "SpokenArabicDigits"] else "stft"
    pretrain_data, train_data, test_data, Y_train, Y_test, is_multivariate, is_instances_classification = load_data(dataset, spectral_representation, visualize=False)
    for function_name in ["hsp", "short-hag"]: # "random_ee", "random_ei", "ip_correct", "anti-oja_fast",  "ip-anti-oja_fast", "hadsp", "desp", "hsp", "short-hag"
        # Get the best trial from the study
        print(function_name)
        study = retrieve_best_model(function_name, dataset, is_multivariate, variate_type = "multi", data_type = "normal")
        
        SRs, pearsons, CEVs, dcors = evaluate_dataset_on_test(
            study, 
            dataset,
            function_name, 
            pretrain_data, 
            train_data, 
            test_data,
            Y_train, 
            Y_test,
            is_instances_classification,
            nb_trials = 4,
            record_metrics=True
        )
        # Create a new DataFrame row
        new_row = pd.DataFrame({
            "dataset": [dataset],
            "function_name": [function_name],
            "spectral_radius_mean": [np.mean(SRs)],
            "spectral_radius_std": [np.std(SRs)],
            "pearson_mean": [np.mean(pearsons)],
            "pearson_std": [np.std(pearsons)],
            "CEVD_mean": [np.mean(CEVs)], 
            "CEVD_std": [np.std(CEVs)],
            "dcor_mean": [np.mean(dcors)],
            "dcor_std": [np.std(dcors)],
        })
    
        # Concatenate the new row to the results DataFrame
        new_results.append(new_row)
        

# Display the DataFrame
print(new_results)

In [ ]:
file_name = "outputs/metrics.csv"

orig = pd.read_csv(file_name).set_index(["dataset", "function_name"])
corr_df = (
    pd.concat(new_results, ignore_index=True)
      .set_index(["dataset", "function_name"])
)

# Add any truly new columns from corr_df into orig
for col in corr_df.columns:
    if col not in orig.columns:
        orig[col] = np.nan

# Keep all existing rows + any new rows from corr_df
merged = orig.reindex(orig.index.union(corr_df.index))

# Update only the columns that were actually computed
for col in corr_df.columns:
    merged.loc[corr_df.index, col] = corr_df[col]

# Save back
merged.reset_index().to_csv(file_name, index=False)

print(
    f"✔ Added {len(set(corr_df.index) - set(orig.index))} new rows "
    f"and updated {len(set(corr_df.index) & set(orig.index))} rows in {file_name}"
)

## Visualize

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl

# ---------- vector-friendly text (Nature likes real text in PDFs) ----------
mpl.rcParams['pdf.fonttype'] = 42      # embed TrueType
mpl.rcParams['ps.fonttype']  = 42
mpl.rcParams['svg.fonttype'] = 'none'  # keep text as text
mpl.rcParams['font.family']  = ['Arial', 'DejaVu Sans']  # Arial if available

# ------------------------------ load data -----------------------------------
file_name = 'outputs/metrics.csv'
data = pd.read_csv(file_name)

datasets_keep = [
    "JapaneseVowels",
    "CatsDogs",
    "FSDD",
#    "SpokenArabicDigits",
#    "SPEECHCOMMANDS",
]

label_map = {
    "JapaneseVowels":     "Japanese\nVowels",
    "CatsDogs":           "Cats vs\nDogs",
    "FSDD":               "FSDD",
#    "SpokenArabicDigits": "Spoken\nArabic\nDigits",
#    "SPEECHCOMMANDS":     "SPEECH\nCOMMANDS",
}

# Stable order present in file
datasets_order = [d for d in datasets_keep if d in data['dataset'].unique()]
dataset_labels = [label_map[d] for d in datasets_order]

data = data[data["dataset"].isin(datasets_order)].copy()
data["dataset"] = data["dataset"].map(label_map)

# Map algorithm names; don't drop rows if mapping missing
data["Algorithm"] = data["function_name"].map(function_mapping).fillna(data["function_name"])

# Keep only algorithms that actually appear, honoring your preferred order
algos_present = list(dict.fromkeys(data["Algorithm"].dropna()))
algos = [a for a in functions_order if a in algos_present] or algos_present

# ------------------------------ layout knobs --------------------------------
fs_axis    = 8.5
fs_tick    = 7.5
width_bar  = min(0.72 / max(1, len(algos)), 0.10)
x          = np.arange(len(datasets_order))

out_dir = "outputs/figures"
os.makedirs(out_dir, exist_ok=True)


# ============================================================================
#  Helper – draws a 1×2 bar-chart figure for two metrics
# ============================================================================
def plot_metric_pair(metric_pair, err_pair, panel_labels, save_path, plot_legend=False):
    fig, axes = plt.subplots(
        2, 1,
        figsize=(9.0, 7.6),
        gridspec_kw=dict(left=0.085, right=0.995, top=0.78, bottom=0.22, wspace=0.40),
        constrained_layout=False,
    )

    handles = labels = None

    for m, (metric, emetric) in enumerate(zip(metric_pair, err_pair)):
        ax = axes[m]

        mean_p = data.pivot_table(index="dataset", columns="Algorithm", values=metric, aggfunc="mean").reindex(index=dataset_labels, columns=algos)
        std_p  = data.pivot_table(index="dataset", columns="Algorithm", values=emetric, aggfunc="mean").reindex(index=dataset_labels, columns=algos)

        for i, alg in enumerate(algos):
            means  = mean_p[alg].to_numpy()
            errors = std_p[alg].to_numpy()
            ax.bar(x + i * width_bar, means, width_bar,
                   label=alg, yerr=errors, capsize=2.5,
                   color=function_colors.get(alg, None),
                   error_kw={"elinewidth": 0.8, "capthick": 0.8})

        ax.set_title(panel_labels[m], loc='left', fontsize=fs_axis,
                     fontweight='bold', pad=2)
        ax.set_ylabel(metric.replace("_", " ").title(), fontsize=fs_axis)
        ax.set_xticks(x + width_bar * (len(algos) - 1) / 2)
        ax.set_xticklabels(dataset_labels, fontsize=fs_tick)
        ax.tick_params(axis="y", labelsize=fs_tick)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.grid(axis='y', linestyle=':', linewidth=0.5, alpha=0.55)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        if m == 0 and plot_legend:
            handles, labels = ax.get_legend_handles_labels()

            fig.legend(handles, labels, title="Algorithm",
                           ncol=min(7, len(labels)), frameon=False,
                           fontsize=fs_tick, title_fontsize=fs_tick,
                           loc="upper center", bbox_to_anchor=(0.5, 0.895),
                           borderaxespad=0.0)

    fig.savefig(save_path, bbox_inches="tight")
    plt.show()


# ============================================================================
#  Figure 1 – panels (a) & (b)
# ============================================================================
plot_metric_pair(
    metric_pair  = ["spectral_radius_mean", "pearson_mean"],
    err_pair     = ["spectral_radius_std",  "pearson_std"],
    panel_labels = ["(a)", "(b)"],
    save_path    = os.path.join(out_dir, "Fig_dim_metrics_ab.pdf"),
    plot_legend  = True
)

# ============================================================================
#  Figure 2 – panels (c) & (d)
# ============================================================================
plot_metric_pair(
    metric_pair  = ["CEVD_mean", "dcor_mean"],
    err_pair     = ["CEVD_std",  "dcor_std"],
    panel_labels = ["(c)", "(d)"],
    save_path    = os.path.join(out_dir, "Fig_dim_metrics_cd.pdf"),
)

# Separability

In [ ]:
from scipy import stats
from numpy import random

# Evaluating
from hag.performances.esn_model_evaluation import (init_reservoir, init_ip_reservoir, init_local_rule_reservoir, init_ip_local_rule_reservoir, init_readout)


import hag.metrics.separability
from hag.models import tanh
activation_function = lambda x : tanh(x)

from importlib import reload
from hag.metrics import (
    inter_intra_class_distance,
    silhouette,
    davies_bouldin,
    calinski_harabasz
)
reload(hag.metrics.separability)

nb_jobs = 10
def evaluate_dataset_on_test_alternative(study, dataset_name, function_name, pretrain_data, train_data, test_data, Y_train, Y_test, is_instances_classification, nb_trials = 8, record_metrics=False):
    # Collect all hyperparameters in a dictionary
    hyperparams = {param_name: param_value for param_name, param_value in study.best_trial.params.items()}
    print(hyperparams)
    leaky_rate = 1
    input_connectivity = 1

    # score for prediction
    if dataset_name == "Sunspot":
        start_step = 30
        end_step = 500
    else:
        start_step = 500
        end_step = 1500
    SLICE_RANGE = slice(start_step, end_step)

    if 'variance_target' not in hyperparams and 'min_variance' in hyperparams:
        hyperparams['variance_target'] = hyperparams['min_variance']
    if not is_instances_classification:
        hyperparams['use_full_instance'] = False

    RIDGE_COEF = 10**hyperparams['ridge']
    
    if function_name in ["hadsp", "desp"]:
        max_partners = np.inf
    
    inter_dists = []
    intra_dists = []
    separability_ratios = []
    sil_scores = []
    dbi_scores = []
    ch_scores  = []

    for i in range(nb_trials):
        print("Trial", i + 1, "of", nb_trials)
        common_index = 1
        if is_instances_classification:
            common_size = pretrain_data[0].shape[common_index]
        else:
            common_size = pretrain_data.shape[common_index]

        # We want the size of the models to be at least network_size
        K = math.ceil(hyperparams['network_size'] / common_size)
        n = common_size * K
        
        if function_name in ["diag_ee", "diag_ei"]:
            use_block = True
        else:
            use_block = False
            
        # UNSUPERVISED PRETRAINING 
        if function_name == "random_ee":
            Win, W, bias = init_matrices(n, input_connectivity, hyperparams['connectivity'],  K, w_distribution=stats.uniform(loc=0, scale=1), use_block=use_block, seed=random.randint(0, 1000))
        else:
            Win, W, bias = init_matrices(n, input_connectivity, hyperparams['connectivity'],  K, w_distribution=stats.uniform(loc=-1, scale=2), use_block=use_block, seed=random.randint(0, 1000))
        bias *= hyperparams['bias_scaling']
        Win *= hyperparams['input_scaling']

        if function_name == "hadsp":
            W, (_, _, _) = run_algorithm(W, Win, bias, hyperparams['leaky_rate'], activation_function, pretrain_data, 
                                     hyperparams['weight_increment'], hyperparams['target_rate'], hyperparams['rate_spread'], "mean_hag", 
                                     multiple_instances=is_instances_classification, 
                                     min_increment = hyperparams['min_increment'], max_increment=hyperparams['max_increment'], use_full_instance=hyperparams['use_full_instance'],
                                     max_partners=max_partners, method="pearson", n_jobs=nb_jobs)
        elif function_name == "desp":
            print("DESP")
            W, (_, _, _) = run_algorithm(W, Win, bias, hyperparams['leaky_rate'], activation_function, pretrain_data, 
                                         hyperparams['weight_increment'], hyperparams['variance_target'], hyperparams['variance_spread'], function_name, 
                                         multiple_instances=is_instances_classification, 
                                         min_increment = hyperparams['min_increment'], max_increment=hyperparams['max_increment'], use_full_instance = hyperparams['use_full_instance'], 
                                         max_partners=max_partners, method = "pearson", 
                                         intrinsic_saturation=hyperparams['intrinsic_saturation'], intrinsic_coef=hyperparams['intrinsic_coef'], 
                                         n_jobs = nb_jobs)
        elif function_name == "short-hag":
            W, (_, _, _) = run_algorithm(W, Win, bias, hyperparams['leaky_rate'], activation_function, pretrain_data,
                                         hyperparams['weight_increment'], hyperparams['target_rate'], hyperparams['rate_spread'], "mean_hag",
                                         multiple_instances=is_instances_classification,
                                         min_increment = 100, max_increment=100, use_full_instance = False,
                                         max_partners=np.inf, method="pearson", n_jobs=nb_jobs)
        elif function_name == "hsp":
            W, (_, _, _) = run_algorithm(W, Win, bias, hyperparams['leaky_rate'], activation_function, pretrain_data,
                                         hyperparams['weight_increment'], hyperparams['target_rate'], hyperparams['rate_spread'],"mean_hag",
                                         multiple_instances=is_instances_classification,
                                         min_increment=1, max_increment=1, use_full_instance=False,
                                         max_partners=np.inf, method="random", n_jobs=nb_jobs)
        elif function_name in ["random_ee", "random_ei", "diag_ee", "diag_ei", "ip_correct", "anti-oja_fast", "ip-anti-oja_fast"]:
            eigen = sparse.linalg.eigs(W, k=1, which="LM", maxiter=W.shape[0] * 20, tol=0.1, return_eigenvectors=False)
            W *= hyperparams['spectral_radius'] / max(abs(eigen))
        else:
            raise ValueError(f"Invalid function: {function_name}")
        
        # unsupervised local rules
        if is_instances_classification:
            unsupervised_pretrain = np.concatenate(pretrain_data).astype(float)
        else:
            unsupervised_pretrain = pretrain_data.astype(float)
        if function_name == "ip_correct":
            reservoir = init_ip_reservoir(W, Win, bias, mu=hyperparams['mu'], sigma=hyperparams['sigma'], learning_rate=hyperparams['learning_rate'],
                                          leaking_rate=hyperparams['leaky_rate'], activation_function=activation_function
                                          )
            _ = reservoir.fit(unsupervised_pretrain, warmup=100)
        elif function_name == "anti-oja_fast":
            reservoir = init_local_rule_reservoir(W, Win, bias, local_rule="anti-oja", eta=hyperparams['oja_eta'],
                                                  synapse_normalization=False, bcm_theta=None,
                                                  leaking_rate=hyperparams['leaky_rate'], activation_function=activation_function,
                                                  )
            _ = reservoir.fit(unsupervised_pretrain, warmup=100)        
        elif function_name == "ip-anti-oja_fast":
            reservoir = init_ip_local_rule_reservoir(W, Win, bias, local_rule="anti-oja", eta=hyperparams['oja_eta'],
                                                      synapse_normalization=False, bcm_theta=None,
                                                      mu=hyperparams['mu'], sigma=hyperparams['sigma'], learning_rate=hyperparams['learning_rate'],
                                                      leaking_rate=hyperparams['leaky_rate'], activation_function=activation_function,
                                                      )
            _ = reservoir.fit(unsupervised_pretrain, warmup=100)
        else:
            reservoir = init_reservoir(W, Win, bias, leaky_rate, activation_function)
        readout = init_readout(ridge_coef=RIDGE_COEF)


        # TRAINING and EVALUATION
        # Step 1: Collect final hidden states
        final_states = []
        for seq in test_data:
            states = reservoir.run(seq)
            final_states.append(states[-1])

        final_states = np.array(final_states)  # (n_test, reservoir_dim)

        # Convert one-hot labels to flat labels
        if Y_test.ndim == 2:
            y_test = np.argmax(Y_test, axis=1)
        else:
            y_test = np.array(Y_test)

        # Step 2: Compute inter/intra class distances
        if is_instances_classification:
            inter_dist, intra_dist, sep_ratio = inter_intra_class_distance(final_states, y_test)
            sil = silhouette(final_states, y_test)
            dbi = davies_bouldin(final_states, y_test)
            ch  = calinski_harabasz(final_states, y_test)

            inter_dists.append(inter_dist)
            intra_dists.append(intra_dist)
            separability_ratios.append(sep_ratio)

            sil_scores.append(sil)
            dbi_scores.append(dbi)
            ch_scores.append(ch)

    return {
        'inter': inter_dists,
        'intra': intra_dists,
        'ratio': separability_ratios,
        'silhouette': sil_scores,
        'davies_bouldin': dbi_scores,
        'calinski_harabasz': ch_scores,
    }


In [ ]:
from hag.performances import retrieve_best_model

datasets = [
    "JapaneseVowels",
    "CatsDogs",
    "FSDD",
#    "SpokenArabicDigits",
#    "SPEECHCOMMANDS",
]


corr_columns = []
for dataset in datasets:
    print(dataset)
    spectral_representation = "mfcc" if dataset in ["CatsDogs", "FSDD", "JapaneseVowels", "SPEECHCOMMANDS", "SpokenArabicDigits"] else "stft"
    pretrain_data, train_data, test_data, Y_train, Y_test, is_multivariate, \
        is_instances_classification = load_data(dataset, spectral_representation, visualize=False)
    for function_name in ["hsp", "short-hag"]: # "random_ee", "random_ei", "ip_correct", "anti-oja_fast", "ip-anti-oja_fast", "hadsp", "desp", "hsp", "short-hag"
        print(function_name)

        study = retrieve_best_model(
            function_name, dataset, is_multivariate,
            variate_type="multi", data_type="normal"
        )

        results = evaluate_dataset_on_test_alternative(
            study, dataset, function_name,
            pretrain_data, train_data, test_data,
            Y_train, Y_test,
            is_instances_classification,
            nb_trials=4,
            record_metrics=True
        )

        # add *one* dict to corr_rows
        corr_columns.append({
            "dataset": dataset,
            "function_name": function_name,
            "inter_dists_mean": np.mean(results['inter']),
            "inter_dists_std":  np.std(results['inter']),
            "intra_dists_mean": np.mean(results['intra']),
            "intra_dists_std":  np.std(results['intra']),
            "separability_ratios_mean": np.mean(results['ratio']),
            "separability_ratios_std":  np.std(results['ratio']),
            "silhouette_mean": np.mean(results['silhouette']),
            "silhouette_std":  np.std(results['silhouette']),
            "davies_bouldin_mean": np.mean(results['davies_bouldin']),
            "davies_bouldin_std":  np.std(results['davies_bouldin']),
            "calinski_harabasz_mean": np.mean(results['calinski_harabasz']),
            "calinski_harabasz_std":  np.std(results['calinski_harabasz']),
        })


In [ ]:
file_name = "outputs/metrics.csv"

orig = pd.read_csv(file_name).set_index(["dataset", "function_name"])
corr_df = pd.DataFrame(corr_columns).set_index(["dataset", "function_name"])

# Add only truly new columns from corr_df into orig
for col in corr_df.columns:
    if col not in orig.columns:
        orig[col] = np.nan

# Keep all original rows plus any new rows from corr_df
merged = orig.reindex(orig.index.union(corr_df.index))

# Update only columns that actually exist in corr_df
for col in corr_df.columns:
    merged.loc[corr_df.index, col] = corr_df[col]

# Write out
merged.reset_index().to_csv(file_name, index=False)

print(
    f"✔ Wrote {len(corr_df)} rows "
    f"(added {len(set(corr_df.index) - set(orig.index))}, "
    f"updated {len(set(corr_df.index) & set(orig.index))}) to {file_name}"
)

## Visualize

In [ ]:
from matplotlib.ticker import LogLocator, LogFormatter


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# ------------------------------------------------------------------
# parameters + data
# ------------------------------------------------------------------
df = pd.read_csv("outputs/metrics.csv")

datasets = [
    "JapaneseVowels", 
    "CatsDogs", 
#    "SpokenArabicDigits", 
    "FSDD", 
#    "SPEECHCOMMANDS"
]
df = df[df["dataset"].isin(datasets)].copy()

label_map = {
    "SpokenArabicDigits": "Spoken\nArabic\nDigits",
    "SPEECHCOMMANDS":     "SPEECH\nCOMMANDS",
    "JapaneseVowels":     "Japanese\nVowels",
}
df["dataset_label"] = df["dataset"].replace(label_map)

dataset_labels = [label_map.get(d, d) for d in datasets]

if "function_mapping" in locals():
    df["Algorithm"] = df["function_name"].map(function_mapping).fillna(df["function_name"])
else:
    df["Algorithm"] = df["function_name"]

if "algos" not in locals():
    algos = list(pd.Index(df["Algorithm"].dropna().unique()))

fs_axis = locals().get("fs_axis", 10)
fs_tick = locals().get("fs_tick", 9)

function_colors = locals().get("function_colors", {a: None for a in algos})

x = np.arange(len(dataset_labels))
width_bar = 0.8 / max(1, len(algos))

os.makedirs("outputs/figures", exist_ok=True)


# ============================================================================
#  Helper – draws a 1×2 bar-chart figure for two metrics
# ============================================================================
def plot_metric_pair(metric_pair, err_pair, panel_labels, save_path, plot_legend=False):
    fig, axes = plt.subplots(
        2, 1,
        figsize=(9.0, 7.6),
        gridspec_kw=dict(left=0.08, right=0.995, top=0.78, bottom=0.22,
                         wspace=0.38),
        constrained_layout=False,
    )

    handles = labels = None

    for m, (metric, emetric) in enumerate(zip(metric_pair, err_pair)):
        ax = axes[m]

        mean_p = (df.pivot_table(index="dataset_label", columns="Algorithm",
                                 values=metric, aggfunc="mean")
                    .reindex(index=dataset_labels, columns=algos))

        if emetric in df.columns:
            std_p = (df.pivot_table(index="dataset_label", columns="Algorithm",
                                    values=emetric, aggfunc="mean")
                       .reindex(index=dataset_labels, columns=algos))
            use_yerr = True
        else:
            std_p = mean_p.copy()
            std_p.loc[:, :] = 0.0
            use_yerr = False

        # --- Davies–Bouldin on log scale ---
        is_db = (metric == "davies_bouldin_mean")
        if is_db:
            ax.set_yscale("log")
            pos_vals = mean_p.to_numpy()
            pos_vals = pos_vals[np.isfinite(pos_vals) & (pos_vals > 0)]
            eps = float(pos_vals.min() * 0.5) if pos_vals.size else 1e-6
            ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=6))
            ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=12))
            ax.yaxis.set_major_formatter(LogFormatter(base=10.0))
            ax.grid(axis='y', which='both', linestyle=':', linewidth=0.5, alpha=0.6)
        else:
            ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
            ax.grid(axis='y', linestyle=':', linewidth=0.5, alpha=0.6)

        for i, alg in enumerate(algos):
            means  = mean_p[alg].to_numpy()
            errors = std_p[alg].to_numpy() if use_yerr else None

            if is_db:
                means = np.where(~np.isfinite(means) | (means <= 0), eps, means)
                if use_yerr and errors is not None:
                    errors = np.clip(errors, 0, means - eps)

            ax.bar(x + i * width_bar, means, width_bar,
                   label=alg, yerr=(errors if use_yerr else None),
                   capsize=2.5 if use_yerr else 0.0,
                   color=function_colors.get(alg, None),
                   error_kw={"elinewidth": 0.8, "capthick": 0.8} if use_yerr else None)

        ax.set_title(panel_labels[m], loc='left', fontsize=fs_axis,
                     fontweight='bold', pad=2)

        nice_name = metric.replace("_", " ").title()
        if metric == "davies_bouldin_mean":
            nice_name = "Davies–Bouldin"
        ax.set_ylabel(nice_name, fontsize=fs_axis)

        ax.set_xticks(x + width_bar * (len(algos) - 1) / 2)
        ax.set_xticklabels(dataset_labels, fontsize=fs_tick)
        ax.tick_params(axis="y", which="both", labelsize=fs_tick)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        if m == 0 and plot_legend:
            handles, labels = ax.get_legend_handles_labels()

            fig.legend(handles, labels, title="Algorithm",
                   ncol=min(7, len(labels)), frameon=False,
                   fontsize=fs_tick, title_fontsize=fs_tick,
                   loc="upper center", bbox_to_anchor=(0.5, 0.895),
                   borderaxespad=0.0)

    fig.savefig(save_path, bbox_inches="tight")
    plt.show()


# ============================================================================
#  Figure 1 – panels (a) & (b)
# ============================================================================
plot_metric_pair(
    metric_pair  = ["separability_ratios_mean", "silhouette_mean"],
    err_pair     = ["separability_ratios_std",   "silhouette_std"],
    panel_labels = ["(a)", "(b)"],
    save_path    = "outputs/figures/Fig_cluster_metrics_ab.pdf",
    plot_legend  = True
)

# ============================================================================
#  Figure 2 – panels (c) & (d)
# ============================================================================
plot_metric_pair(
    metric_pair  = ["davies_bouldin_mean", "calinski_harabasz_mean"],
    err_pair     = ["davies_bouldin_std",  "calinski_harabasz_std"],
    panel_labels = ["(c)", "(d)"],
    save_path    = "outputs/figures/Fig_cluster_metrics_cd.pdf",
)

# ============================================================================
#  Figure 3 – panel (e): cumulative ranking
# ============================================================================
metrics = [
    "separability_ratios_mean",
    "silhouette_mean",
    "davies_bouldin_mean",
    "calinski_harabasz_mean",
]

if "rank_sums" not in locals():
    lower_is_better = {"davies_bouldin_mean"}
    ranks_per_alg = pd.Series(0.0, index=algos, dtype=float)

    for metric in metrics:
        pivot = (df.pivot_table(index="dataset_label", columns="Algorithm",
                                values=metric, aggfunc="mean")
                   .reindex(index=dataset_labels, columns=algos))
        if pivot.empty:
            continue
        ascending = metric in lower_is_better
        ranks = pivot.rank(axis=1, ascending=ascending, method="average")
        ranks_per_alg = ranks_per_alg.add(ranks.sum(skipna=True), fill_value=0.0)

    rank_sums = ranks_per_alg.sort_values()

RANK_WIDTH_FRAC = 0.55
fig_e, ax_rank = plt.subplots(
    figsize=(9.0, 3.8),
    gridspec_kw=dict(left=0.08, right=0.995, top=0.82, bottom=0.28),
    constrained_layout=False,
)

colors = [function_colors.get(alg, None) for alg in rank_sums.index]
bars = ax_rank.bar(rank_sums.index, rank_sums.values,
                   edgecolor='black', color=colors, linewidth=0.8)

ax_rank.set_title("(e)", loc='left', fontsize=fs_axis, fontweight='bold', pad=2)
ax_rank.set_ylabel("Cumulative Rank", fontsize=fs_axis)
ax_rank.spines['top'].set_visible(False)
ax_rank.spines['right'].set_visible(False)
ax_rank.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
ax_rank.set_axisbelow(True)
ax_rank.tick_params(axis="y", labelsize=fs_tick)
ax_rank.set_xticklabels(rank_sums.index, rotation=45, ha="right", fontsize=fs_tick)

ymax = float(rank_sums.max() if len(rank_sums) else 0)
for bar, val in zip(bars, rank_sums.values):
    ax_rank.text(
        bar.get_x() + bar.get_width() * 0.5,
        bar.get_height() + (0.02 * ymax if ymax > 0 else 0.2),
        f"{int(round(val))}",
        ha='center', va='bottom', fontsize=fs_tick,
    )

fig_e.savefig("outputs/figures/Fig_cluster_metrics_e.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd

# Lire le CSV
df = pd.read_csv("outputs/metrics.csv")

# Arrondir toutes les colonnes numériques à 5 décimales
df = df.round(5)

# Sauvegarder le fichier modifié (optionnel)
df.to_csv("outputs/metrics_rounded.csv", index=False)

print("Fichier arrondi sauvegardé dans outputs/metrics_rounded.csv")

## Ranking

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load and filter data
data = pd.read_csv("outputs/metrics.csv")
datasets = ["JapaneseVowels", "CatsDogs", "SpokenArabicDigits", "FSDD", "SPEECHCOMMANDS"]
df = data[data["dataset"].isin(datasets)].copy()

# Map function names to algorithm names
df["Algorithm"] = df["function_name"].map(function_mapping)

# Metrics used
metrics = [
    "separability_ratios_mean",
    "silhouette_mean",
    "davies_bouldin_mean",
    "calinski_harabasz_mean"
]

# Average metrics per dataset-algorithm combination
agg = df.groupby(["dataset", "Algorithm"])[metrics].mean().reset_index()

algorithms = ["E-ESN", "ESN", "IP", "Anti-Oja", "IP +\nAnti-Oja", "mean HAG", "variance HAG"]
rank_sums = pd.Series(0, index=algorithms)

# Compute ranks and sum across all dataset × metric combinations
for ds in datasets:
    sub = agg[agg["dataset"] == ds].set_index("Algorithm")
    for metric in metrics:
        ascending = metric == "davies_bouldin_mean"  # lower is better for DB, higher for others
        ranks = sub[metric].rank(ascending=ascending, method="min")
        rank_sums += ranks.reindex(algorithms)

# Sort by cumulative rank sums
rank_sums = rank_sums.sort_values()

# --------------
# Plot

# Plotting cumulative rank sums (polished for publication)
plt.figure(figsize=(8, 5))
colors = [function_colors[alg] for alg in rank_sums.index]
bars = plt.bar(rank_sums.index, rank_sums.values, edgecolor='black', color=colors, linewidth=0.8)

# Clean up spines 
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add horizontal grid lines for readability
ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
ax.set_axisbelow(True)

# Labels and title in bold
plt.ylabel("Cumulative Rank Sum", fontsize=12, fontweight='bold')

# Ticks styling
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(fontsize=10)

# Annotate bars with integer labels
ymax = rank_sums.max()
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() * 0.5,
        height + ymax * 0.02,
        f"{int(height)}",
        ha='center',
        va='bottom',
        fontsize=10
    )

plt.tight_layout()
plt.show()

# Final matrix

In [ ]:
from hag.performances import retrieve_best_model, init_ip_reservoir, init_reservoir, init_local_rule_reservoir, init_ip_local_rule_reservoir
from hag.models import init_matrices
from hag.hag import run_algorithm
from scipy import sparse

# List of datasets
classification = [
    "JapaneseVowels",
    "CatsDogs",
    "FSDD",
    "SpokenArabicDigits",
    "SPEECHCOMMANDS",
]

prediction = [
    "MackeyGlass",
    "Lorenz",
    "Sunspot_daily",
]
datasets=classification


# Initialize lists to store results and max values
Ws = []
titles = []
max_values = []

leaky_rate = 1
input_connectivity = 1

# Loop through datasets and function names to compute W matrices and find global vmax
for dataset in datasets:
    print(dataset)
    spectral_representation = "mfcc" if dataset in ["CatsDogs", "FSDD", "JapaneseVowels", "SPEECHCOMMANDS", "SpokenArabicDigits"] else "stft"
    pretrain_data, train_data, test_data, Y_train, Y_test, is_multivariate, is_instances_classification = load_data(dataset, spectral_representation)

    # "random_ee", "random_ei", "ip_correct", "anti-oja_fast", "ip-anti-oja_fast",  "hadsp", "desp", "diag_ee", "diag_ei"
    for function_name in ["random_ee", "random_ei", "ip_correct", "anti-oja_fast", "ip-anti-oja_fast",  "hadsp", "desp"]: #"diag_ee", "diag_ei",
        print(function_name)
        # Get the best trial from the study
        study = retrieve_best_model(function_name, dataset, is_multivariate, variate_type = "multi", data_type = "normal", prefix="tpe", db_dir="hag/hpo/legacy_studies")
        hyperparams = {param_name: param_value for param_name, param_value in study.best_trial.params.items()}
        print(hyperparams)

        if 'variance_target' not in hyperparams and 'min_variance' in hyperparams:
            hyperparams['variance_target'] = hyperparams['min_variance']
        if not is_instances_classification:
            hyperparams['use_full_instance'] = False
    
        if function_name in ["hadsp", "desp"]:
            max_partners = np.inf
        
        common_index = 1
        if is_instances_classification:
            common_size = pretrain_data[0].shape[common_index]
        else:
            common_size = pretrain_data.shape[common_index]

        # We want the size of the models to be at least network_size
        K = math.ceil(hyperparams["network_size"] / common_size)
        n = common_size * K
        
        if function_name in ["diag_ee", "diag_ei"]:
            use_block = True
        else:
            use_block = False
            
        # UNSUPERVISED PRETRAINING 
        if function_name in ["random_ee", "diag_ee"]:
            Win, W, bias = init_matrices(n, input_connectivity, hyperparams['connectivity'],  K, w_distribution=stats.uniform(loc=0, scale=1), use_block=use_block, seed=random.randint(0, 1000), random_projection_experiment=False)
        else:
            Win, W, bias = init_matrices(n, input_connectivity, hyperparams['connectivity'],  K, w_distribution=stats.uniform(loc=-1, scale=2), use_block=use_block, seed=random.randint(0, 1000), random_projection_experiment=False)
        bias *= hyperparams['bias_scaling']
        Win *= hyperparams['input_scaling']

        if function_name in ("hadsp", "mean_hag"):
            W, (_, _, _) = run_algorithm(W, Win, bias, hyperparams['leaky_rate'], activation_function, pretrain_data,
                                     hyperparams['weight_increment'], hyperparams['target_rate'], hyperparams['rate_spread'], "mean_hag",
                                     multiple_instances=is_instances_classification,
                                     min_increment = hyperparams['min_increment'], max_increment=hyperparams['max_increment'], use_full_instance=hyperparams['use_full_instance'],
                                     max_partners=np.inf, method="pearson", n_jobs=1)
        elif function_name in ("desp", "var_hag"):
            W, (_, _, _) = run_algorithm(W, Win, bias, hyperparams['leaky_rate'], activation_function, pretrain_data,
                                         hyperparams['weight_increment'], hyperparams['variance_target'], hyperparams['variance_spread'], "var_hag",
                                         multiple_instances=is_instances_classification,
                                         min_increment = hyperparams['min_increment'], max_increment=hyperparams['max_increment'], use_full_instance = hyperparams['use_full_instance'],
                                         max_partners=np.inf, method = "pearson",
                                         intrinsic_saturation=hyperparams['intrinsic_saturation'], intrinsic_coef=hyperparams['intrinsic_coef'],
                                         n_jobs = 1)
        elif function_name == "short-hag":
            W, (_, _, _) = run_algorithm(W, Win, bias, hyperparams['leaky_rate'], activation_function, pretrain_data,
                                         hyperparams['weight_increment'], hyperparams['target_rate'], hyperparams['rate_spread'], "mean_hag",
                                         multiple_instances=is_instances_classification,
                                         min_increment = 1, max_increment=1, use_full_instance = False,
                                         max_partners=np.inf, method="hebbian", n_jobs=1)
        elif function_name == "hsp":
            W, (_, _, _) = run_algorithm(W, Win, bias, hyperparams['leaky_rate'], activation_function, pretrain_data,
                                         hyperparams['weight_increment'], hyperparams['target_rate'], hyperparams['rate_spread'],"mean_hag",
                                         multiple_instances=is_instances_classification,
                                         min_increment=100, max_increment=100, use_full_instance=False,
                                         max_partners=np.inf, method="random", n_jobs=1)
        elif function_name in ["random_ee", "random_ei", "diag_ee", "diag_ei", "ip_correct", "anti-oja_fast", "ip-anti-oja_fast"]:
            eigen = sparse.linalg.eigs(W, k=1, which="LM", maxiter=W.shape[0] * 20, tol=0.1, return_eigenvectors=False)
            W *= hyperparams['spectral_radius'] / max(abs(eigen))
        else:
            raise ValueError(f"Invalid function: {function_name}")

        # unsupervised local rules
        if is_instances_classification:
            unsupervised_pretrain = np.concatenate(pretrain_data).astype(float)
        else:
            unsupervised_pretrain = pretrain_data.astype(float)
        if function_name == "ip_correct":
            reservoir = init_ip_reservoir(W, Win, bias, mu=hyperparams['mu'], sigma=hyperparams['sigma'], learning_rate=hyperparams['learning_rate'],
                                          leaking_rate=hyperparams['leaky_rate'])
            _ = reservoir.fit(unsupervised_pretrain, warmup=100)
        elif function_name == "anti-oja_fast":
            reservoir = init_local_rule_reservoir(W, Win, bias, local_rule="anti-oja", eta=hyperparams['oja_eta'],
                                                  synapse_normalization=False, bcm_theta=None,
                                                  leaking_rate=hyperparams['leaky_rate'], activation_function=activation_function,
                                                  )
            _ = reservoir.fit(unsupervised_pretrain, warmup=100)
        elif function_name == "ip-anti-oja_fast":
            reservoir = init_ip_local_rule_reservoir(W, Win, bias, local_rule="anti-oja", eta=hyperparams['oja_eta'],
                                                      synapse_normalization=False, bcm_theta=None,
                                                      mu=hyperparams['mu'], sigma=hyperparams['sigma'], learning_rate=hyperparams['learning_rate'],
                                                      leaking_rate=hyperparams['leaky_rate'])
            _ = reservoir.fit(unsupervised_pretrain, warmup=100)
        else:
            reservoir = init_reservoir(W, Win, bias, leaky_rate, activation_function)

        W = reservoir.W
            

        # Store W matrix and corresponding title
        Ws.append(W)
        titles.append(
            f"{dataset} - "
            f"{function_mapping.get(function_name, function_name)}"
        )
        max_values.append(np.max(W))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import matplotlib as mpl
from pathlib import Path

# --- use the same raw function order you trained with ---
functions_order = [
    "random_ee",
    "random_ei",
    "ip_correct",
    "anti-oja_fast",
    "ip-anti-oja_fast",
    "hadsp",
    "desp",
]

n_datasets = len(datasets)
n_functions = len(functions_order)

# Safety check: did we compute one W per (dataset, function)?
expected = n_datasets * n_functions
if len(Ws) != expected:
    raise ValueError(f"Mismatch: have {len(Ws)} W matrices for {n_datasets} datasets × {n_functions} functions (expected {expected}). Ensure functions_order matches the training loop.")

# --- global values across all Ws ---
all_values = []
for W in Ws:
    if hasattr(W, "toarray"):
        W = W.toarray()
    W = np.asarray(W)
    all_values.append(W.ravel())

all_values = np.concatenate(all_values)
all_values = all_values[np.isfinite(all_values)]

if all_values.size == 0:
    raise ValueError("No finite values found in Ws.")

# --- robust global range (clip outliers) ---
abs_vals = np.abs(all_values)
gmax = np.quantile(abs_vals, 0.995)

if gmax == 0:
    gmax = 1e-8

linthresh = max(gmax * 0.02, 1e-8)

norm = mcolors.SymLogNorm(linthresh=linthresh, vmin=-gmax, vmax=gmax, base=10)
cmap = plt.get_cmap("RdBu_r")

# embed TrueType fonts so PDF text is selectable
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# --- figure ---
fig, axes = plt.subplots(n_datasets, n_functions, figsize=(n_functions * 3, n_datasets * 3), sharex=True, sharey=True)

# Ensure axes is 2D even if a dimension is 1
if n_datasets == 1 and n_functions == 1:
    axes = np.array([[axes]])
elif n_datasets == 1:
    axes = np.array([axes])
elif n_functions == 1:
    axes = np.array([[ax] for ax in axes])

im = None

for i in range(n_datasets):
    for j in range(n_functions):
        idx = i * n_functions + j

        W = Ws[idx]
        if hasattr(W, "toarray"):
            W = W.toarray()
        W = np.asarray(W)

        im = axes[i, j].imshow(W, cmap=cmap, norm=norm, interpolation="nearest")

        # Overlay zeros in white
        zero_mask = W == 0
        axes[i, j].imshow(np.ma.masked_where(~zero_mask, zero_mask), cmap=mcolors.ListedColormap(["white"]), interpolation="nearest")

        axes[i, j].set_xlabel("")
        axes[i, j].set_ylabel("")

        # Dataset label: raw name -> readable name
        if j == 0:
            dataset_label = dataset_label_map.get(datasets[i], datasets[i])
            axes[i, j].text(-0.3, 0.5, dataset_label, rotation=90, transform=axes[i, j].transAxes, ha="center", va="center", fontsize=15)

        # Function title: raw function name -> readable name
        if i == 0:
            function_label = function_mapping.get(functions_order[j], functions_order[j])
            axes[i, j].set_title(function_label, fontsize=15)

# Shared labels
fig.text(0.5, 0.005, "Neurons", ha="center", va="center", fontsize=14)
fig.text(0.015, 0.5, "Neurons", ha="center", va="center", rotation="vertical", fontsize=14)

fig.tight_layout()

# Single colorbar
cbar = fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.03, pad=0.01)
cbar.set_label("Recurrent weight strength", fontsize=12)

tick_vals = np.array([-gmax, -gmax / 10, -linthresh, 0, linthresh, gmax / 10, gmax])
cbar.set_ticks(tick_vals)
cbar.ax.set_yticklabels([f"{t:.2g}" for t in tick_vals])

if "classification" in locals() and set(datasets).issubset(set(classification)):
    out_base = "outputs/figures/connectivity_matrices_classification"
elif "prediction" in locals() and set(datasets).issubset(set(prediction)):
    out_base = "outputs/figures/connectivity_matrices_forecasting"
else:
    out_base = "outputs/figures/connectivity_matrices"

Path(out_base).parent.mkdir(parents=True, exist_ok=True)

fig.savefig(f"{out_base}.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

functions_order = [
    "random_ee",
    "random_ei",
    "ip_correct",
    "anti-oja_fast",
    "ip-anti-oja_fast",
    "hadsp",
    "desp",
]

rank_tol = None      # None = NumPy default tolerance
zero_tol = 1e-12     # tolerance for sparsity


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def as_dense_array(W):
    """Convert sparse/dense matrix to a NumPy array."""
    if hasattr(W, "toarray"):
        W = W.toarray()
    return np.asarray(W, dtype=float)


def matrix_rank_info(W, tol=None):
    """Return numerical rank and normalized rank."""
    W = as_dense_array(W)
    rank = np.linalg.matrix_rank(W, tol=tol)
    rank_ratio = rank / min(W.shape)
    return rank, rank_ratio


def sparsity_info(W, zero_tol=1e-12):
    """Return fraction of entries close to zero."""
    W = as_dense_array(W)
    return np.mean(np.abs(W) <= zero_tol)


def symmetry_closeness_info(W):
    """
    Scalar measure of closeness to a symmetric matrix.

    asymmetry = ||W - W.T||_F / ||W||_F
    symmetry_closeness = 1 / (1 + asymmetry)

    symmetry_closeness:
        1.0  = perfectly symmetric
        ~0.5 = moderately asymmetric
        near 0 = very asymmetric
    """
    W = as_dense_array(W)

    if W.shape[0] != W.shape[1]:
        return {
            "is_square": False,
            "asymmetry": np.nan,
            "symmetry_closeness": np.nan,
        }

    W_norm = np.linalg.norm(W, ord="fro")
    asym_norm = np.linalg.norm(W - W.T, ord="fro")

    if W_norm == 0:
        asymmetry = 0.0
    else:
        asymmetry = asym_norm / W_norm

    symmetry_closeness = 1.0 / (1.0 + asymmetry)

    return {
        "is_square": True,
        "asymmetry": asymmetry,
        "symmetry_closeness": symmetry_closeness,
    }


# ---------------------------------------------------------------------
# Safety checks
# ---------------------------------------------------------------------

n_datasets = len(datasets)
n_functions = len(functions_order)
expected = n_datasets * n_functions

if len(Ws) != expected:
    raise ValueError(
        f"Mismatch: have {len(Ws)} matrices, expected "
        f"{n_datasets} datasets × {n_functions} functions = {expected}."
    )


# ---------------------------------------------------------------------
# Compute metrics
# ---------------------------------------------------------------------

rows = []

for dataset_idx, dataset_name in enumerate(datasets):
    for function_idx, function_name in enumerate(functions_order):
        idx = dataset_idx * n_functions + function_idx
        W = as_dense_array(Ws[idx])

        rank, rank_ratio = matrix_rank_info(W, tol=rank_tol)
        sparsity = sparsity_info(W, zero_tol=zero_tol)
        sym = symmetry_closeness_info(W)

        rows.append({
            "dataset": dataset_name,
            "function": function_name,
            "shape": f"{W.shape[0]} × {W.shape[1]}",
            "rank": rank,
            "rank_ratio": rank_ratio,
            "sparsity": sparsity,
            "is_square": sym["is_square"],
            "asymmetry": sym["asymmetry"],
            "symmetry_closeness": sym["symmetry_closeness"],
        })

matrix_metrics = pd.DataFrame(rows)


# ---------------------------------------------------------------------
# Display full table
# ---------------------------------------------------------------------

display(
    matrix_metrics.style.format({
        "rank_ratio": "{:.3f}",
        "sparsity": "{:.3f}",
        "asymmetry": "{:.3e}",
        "symmetry_closeness": "{:.4f}",
    })
)


# ---------------------------------------------------------------------
# Optional: compact pivot tables
# ---------------------------------------------------------------------

display(
    matrix_metrics.pivot(
        index="dataset",
        columns="function",
        values="rank_ratio",
    ).style.format("{:.3f}")
)

display(
    matrix_metrics.pivot(
        index="dataset",
        columns="function",
        values="symmetry_closeness",
    ).style.format("{:.4f}")
)

# Cumulated visualisation

In [ ]:
import os
import pandas as pd
import xml.etree.ElementTree as ET
from svgpath2mpl import parse_path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.path import Path
from matplotlib.font_manager import FontProperties
from matplotlib.textpath import TextPath
from matplotlib.markers import MarkerStyle

def make_colorable_marker(svg_path, scale=1.0):
    tree = ET.parse(svg_path)
    root = tree.getroot()
    ns = {"svg": "http://www.w3.org/2000/svg"}
    d_list = [p.attrib["d"] for p in root.findall(".//svg:path", ns)]
    full_d = " ".join(d_list)
    p = parse_path(full_d)
    p.vertices[:, 1] *= -1
    p.vertices -= p.vertices.mean(axis=0)
    p.vertices *= scale
    return MarkerStyle(p)

def make_text_marker(char, family="DejaVu Sans", scale=1.0):
    """Create a centered, scalable marker from a single glyph."""
    fp = FontProperties(family=family)
    tp = TextPath((0, 0), char, prop=fp, size=1.0)
    verts = tp.vertices.copy()
    verts -= verts.mean(axis=0)
    verts *= scale
    path = Path(verts, tp.codes)
    return MarkerStyle(path)

# ---------- vector-friendly text (keep text as text in PDF/SVG) ----------
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = ["Arial", "DejaVu Sans"]

file_name = "outputs/metrics.csv"
performance_file = "outputs/test_results/test_results_classification.csv"
data = pd.read_csv(file_name)
performance = pd.read_csv(performance_file)

# ---------- keep the following datasets -------------------------
datasets_keep = [
    "JapaneseVowels",
    "CatsDogs",
    # "SpokenArabicDigits",
    "FSDD",
    # "SPEECHCOMMANDS",
]
data = data[data["dataset"].isin(datasets_keep)]

# ---------- map functions and datasets → human-readable ----------
data["dataset"] = (
    data["dataset"]
    .str.replace("SpokenArabicDigits", "Spoken\nArabic\nDigits")
    .str.replace("SPEECHCOMMANDS", "SPEECH\nCOMMANDS")
    .str.replace("JapaneseVowels", "Japanese\nVowels")
)
data["Algorithm"] = data["function_name"].map(function_mapping)

# ---------- keep wanted algorithms only -------------------------
algorithms_keep = [
    "E-ESN",
    "ESN",
    "IP",
    "Anti-Oja",
    "IP +\nAnti-Oja",
    "mean HAG",
    "variance HAG",
    "HSP",
    "short HAG",
]

data = data[data["Algorithm"].isin(algorithms_keep)]
function_colors = {k: v for k, v in function_colors.items() if k in algorithms_keep}
functions_order = [f for f in functions_order if f in algorithms_keep]

# ---------- marker for every dataset ----------------------------
dataset_marker = {
    "CatsDogs": make_text_marker("🐱", family="DejaVu Sans", scale=0.020),
    "Japanese\nVowels": make_text_marker("¥", family="DejaVu Sans", scale=0.020),
    "Spoken\nArabic\nDigits": make_text_marker("ب", family="Geeza Pro", scale=0.020),
    "FSDD": make_text_marker("5", family="DejaVu Sans", scale=0.020),
    "SPEECH\nCOMMANDS": make_colorable_marker("hag/metrics/icons/microphone.svg", scale=0.015),
}

display_datasets_keep = (
    pd.Series(datasets_keep)
    .str.replace("SpokenArabicDigits", "Spoken\nArabic\nDigits")
    .str.replace("SPEECHCOMMANDS", "SPEECH\nCOMMANDS")
    .str.replace("JapaneseVowels", "Japanese\nVowels")
    .tolist()
)

dataset_marker = {
    ds: mk for ds, mk in dataset_marker.items()
    if ds in display_datasets_keep
}

# ---------- aggregate to one point per (Algorithm, Dataset) ----
group = data.groupby(["Algorithm", "dataset"]).mean(numeric_only=True).reset_index()

# ---------- merge in performance -------------------------------
performance["Algorithm"] = performance["Function"].map(function_mapping)
performance["dataset"] = (
    performance["Dataset"]
    .str.replace("SpokenArabicDigits", "Spoken\nArabic\nDigits")
    .str.replace("SPEECHCOMMANDS", "SPEECH\nCOMMANDS")
    .str.replace("JapaneseVowels", "Japanese\nVowels")
)
performance["score"] = (
    performance["Average Score"]
    .str.replace("%", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

performance = performance[
    performance["Algorithm"].isin(algorithms_keep)
    & performance["dataset"].isin(display_datasets_keep)
]

group = pd.merge(
    group,
    performance[["Algorithm", "dataset", "score"]],
    on=["Algorithm", "dataset"],
    how="left",
)

# Normalize performance per dataset and size the markers
group["score"] = group.groupby("dataset")["score"].transform(
    lambda x: (x - x.min()) / (x.max() - x.mean() if (x.max() - x.min()) == 0 else (x.max() - x.min()))
)
group["size"] = group["score"] * 1000

# ---------------------------------------------------------------
# 3) Scatter plot : decorrelation vs separability_score
# ---------------------------------------------------------------
metric_x = "CEVD_mean"
metric_y = "intra_dists_mean"

group["x_norm"] = group.groupby("dataset")[metric_x].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)
group["y_norm"] = group.groupby("dataset")[metric_y].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

for _, row in group.iterrows():
    alg = row["Algorithm"]
    ds = row["dataset"]
    ax.scatter(
        row[metric_x],
        row[metric_y],
        marker=dataset_marker.get(ds, "o"),
        s=row["size"],
        edgecolor="none",
        facecolor=function_colors[alg],
        linewidths=1.2,
        zorder=3,
    )

# ---------- legends ---------------------------------------------
alg_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markersize=12,
        markeredgecolor="none",
        markerfacecolor=function_colors[alg],
        label=alg,
    )
    for alg in functions_order
    if alg in function_colors
]

ds_handles = [
    Line2D(
        [0], [0],
        marker=mk,
        linestyle="",
        markersize=14,
        markeredgecolor="none",
        markerfacecolor="black",
        label=ds.replace("\n", " "),
    )
    for ds, mk in dataset_marker.items()
]

legend1 = ax.legend(handles=alg_handles, title="Algorithm", loc="upper left", frameon=False)
ax.add_artist(legend1)
ax.legend(handles=ds_handles, title="Dataset", loc="center right", frameon=False)

# ---------- cosmetics -------------------------------------------
for side in ("left", "bottom"):
    ax.spines[side].set_visible(True)
    ax.spines[side].set_linewidth(1.3)
    ax.spines[side].set_color("black")

ax.xaxis.set_ticks_position("bottom")
ax.yaxis.set_ticks_position("left")

ax.set_xlabel(metric_x.replace("_", " ").title(), fontsize=16)
ax.set_ylabel(metric_y.replace("_", " ").title(), fontsize=16)
ax.grid(alpha=0.3)
plt.tight_layout()

# ---------- SAVE VECTOR OUTPUTS --------------------------------
out_dir = "outputs/figures"
os.makedirs(out_dir, exist_ok=True)
pdf_path = os.path.join(out_dir, "Fig_scatter_CEVD_vs_Intra.pdf")

fig.savefig(pdf_path, bbox_inches="tight")

plt.show()
print(f"Saved:\n - {pdf_path}\n")